# Summarising results

This notebook summarises results into a Pandas dataframe which is then
reformatted into a table suitable for publishing.

In [1]:
from qaoa_parameter_setting.utils.summary import SummaryTable, formatted_styler_for
import pandas as pd
from datetime import datetime, timezone

In [ ]:
# TABLE_JSON: str | None = "summary_tables.json"
TABLE_JSON: str | None = None
table = SummaryTable(TABLE_JSON)

## Setup table with training data and min-max cuts.

In [3]:
if TABLE_JSON is None:
    # Add training data. Only the "best" results are kept, per graph, trainer config
    # file, and depth.
    table.add_data("../data/training/random_regular")
    table.add_data("../data/training/heavy_hex")
    table.add_data("../data/training/line_to_full")
    table.add_data("../data/training/erdos_renyi")

In [4]:
if TABLE_JSON is None:
    # Add min- and max-cut data. If some graph instances do not have a minmax_cuts
    # entry, an error will be thrown later.
    table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
    table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
    table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
    table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")

In [5]:
if TABLE_JSON is None:
    # Register all methods. This just keeps track of the methods so we can
    # identify missing training data.
    table.add_methods("../methods/")

## Identify missing min- and max-cuts data

In [6]:
missing_minmax_cuts = table.missing_minmax_cuts()
if len(missing_minmax_cuts) == 0:
    print("All minmax_cuts data accounted for.")
else:
    print(
        "The following graphs are missing min- and max-cuts data. "
        + "Generate them with compute_min_max_for_graph.py"
    )
    for _graph in missing_minmax_cuts:
        print("- {}".format(_graph))

All minmax_cuts data accounted for.


In [ ]:
table.save_data("summary_tables.json", overwrite=True)

## Get raw data table

In [8]:
# This is the _raw_ table with all results
df: pd.DataFrame = table.to_dataframe()
df

,graph_type,graph_idx,num_nodes,trainer_config,depth,edge_probability,regular_degree,heavy_hex_rows,heavy_hex_cols,num_swap_layers,graph_key,energy,qaoa_angles,result_filename,trainer,evaluation,approximation_ratio
0,random_regular,10,10,FA_SV_opt.json,5,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,5.054196,"[0.5185304937079699, 0.40338118162911696, 0.36...",20250818_100919_000N10R4R_MC_FA_SV_opt_5.json,ScipyTrainer,SV,0.940887
1,random_regular,10,10,F_SV.json,1,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,2.378527,"[0.31879448703883545, 0.4709011035214979]",20250818_100929_000N10R4R_MC_F_SV_5.json,ScipyTrainer,SV,0.773658
2,random_regular,10,10,F_SV.json,2,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,3.247829,"[0.43525303748806166, 0.24541946792775127, 0.3...",20250818_100929_000N10R4R_MC_F_SV_5.json,ScipyTrainer,SV,0.827989
3,random_regular,10,10,F_SV.json,3,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,3.982844,"[0.44499703847182537, 0.34019290858643997, 0.2...",20250818_100929_000N10R4R_MC_F_SV_5.json,ScipyTrainer,SV,0.873928
4,random_regular,10,10,F_SV.json,4,NaN,4.0,NaN,NaN,NaN,000_10nodes_random4regular.json,4.615799,"[0.47848174579525693, 0.37169334024352796, 0.3...",20250818_100929_000N10R4R_MC_F_SV_5.json,ScipyTrainer,SV,0.913487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11033,erdos_renyi,40,40,TQA_MPS_opt.json,8,0.2,NaN,NaN,NaN,NaN,009_40nodes_erdosrenyi20percent.json,32.368283,"[1.0648786525451368, 1.2628147973906527, 0.697...",20251005_112935_009N40ER20_MC_TQA_MPS_optBD16_...,ScipyTrainer,MPS,0.956343
11034,erdos_renyi,40,40,TQA_PP_opt.json,2,0.2,NaN,NaN,NaN,NaN,009_40nodes_erdosrenyi20percent.json,22.104527,"[0.43583634403500415, 0.30296891060824194, 0.2...",20251005_113714_009N40ER20_MC_TQA_PP_optMW4_2....,ScipyTrainer,PP,0.876779
11035,erdos_renyi,40,40,TQA_PP_opt.json,4,0.2,NaN,NaN,NaN,NaN,009_40nodes_erdosrenyi20percent.json,26.909514,"[0.854792312399733, 0.4748883188975097, 0.4390...",20251005_122137_009N40ER20_MC_TQA_PP_optMW4_4....,ScipyTrainer,PP,0.914027
11036,erdos_renyi,40,40,TQA_PP_opt.json,6,0.2,NaN,NaN,NaN,NaN,009_40nodes_erdosrenyi20percent.json,31.191838,"[0.9352235694985224, 0.8649942100734089, 0.531...",20251005_131926_009N40ER20_MC_TQA_PP_optMW4_6....,ScipyTrainer,PP,0.947224


## Get formatted and pivoted table and styler

With Pandas, dataframes are formatted with Stylers.
`qaoa_parameter_setting.utils.summary_table.formatted_styler_for` automatically
stylises the Styler and returns the pivoted dataframe and Styler.

### Example summary table with `"text"` formatting

The following cell formats the summary results into a single table suitable for
Jupyter notebooks. Later cells will format the table for LaTeX and save them to
a file.

#### MAXCUT Approximation Ratio

In [17]:
pivot, styler = formatted_styler_for(
    table,
    depth=10,
    agg_values="approximation_ratio",
    with_fancy_values=True,
    cmap="Greens",
    precision=2,
    missing_data_str="-",
    target_format="text",
)
styler

#### Number of instances and nodes

In [18]:
pivot, styler = formatted_styler_for(
    table,
    depth=10,
    agg_values="num_instances",
    with_fancy_values=True,
    cmap="Greens",
    precision=0,
    missing_data_str="-",
    target_format="text",
)
styler

## Create and save tables for LaTeX

Here we format tables into LaTeX files and save them into separate `.tex` files
suitable for inclusion in a paper. Files are saved with the current date and
time for tracking changes. These can be compiled into a preview of all tables
with `pdflatex summary_tables.tex`

In [ ]:
now = datetime.now(tz=timezone.utc)
generated_on_str = "% Generated on {now:%Y-%m-%d} at {now:%H:%M:%S} UTC\n".format(
    now=now
)
# Open a summary tables.tex file for previewing all tables.
with open("tables.tex", "w") as f:
    # Write date and time to tables.tex
    f.write(generated_on_str)

    # Iterate over all depths, sorted so they're included in increasing order in
    # tables.tex
    for depth in sorted(int(x) for x in table.to_dataframe()["depth"].unique()):
        # Write a section title.
        _ = f.write("\n\\section{{Tables for depth $P={}$}}\n".format(depth))

        # For each value to be plotted
        for values, value_label in [
            ("num_instances", "Number of Instances (num. nodes in brackets)"),
            (
                "approximation_ratio",
                r"Avg. Approx. Ratio $\pm$ standard deviation",
            ),
        ]:
            # This is the filename for this table.
            _latex_filename = "table_p{:02}_{}.tex".format(int(depth), values)

            # Get the styler
            _, styler = formatted_styler_for(
                table,
                depth=depth,
                agg_values=values,
                with_fancy_values=True,
                cmap="Greens",
                precision=0 if values == "num_instances" else 5,
                missing_data_str="-",
                target_format="siunitx",
            )

            # Save table to separate LaTeX file
            with open(_latex_filename, "w") as f_table:
                _ = f_table.write(generated_on_str)
                _ = f_table.write(
                    "% Data is {value_label} for depth P={depth}.\n".format(
                        value_label=value_label, depth=depth
                    )
                )
                styler.to_latex(
                    f_table,
                    convert_css=True,
                    hrules=True,
                    clines="skip-last;data",
                )
            _ = f.write(
                r"""
\begin{{table}}[H]
    \centering
    \input{{{filename}}}
    \caption{{\textbf{{{value_label} for $P={depth}$.}} Graph types are Erdos Renyi (ER), Heavy-Hex (HH), Line-to-Full (L2F), and Random Regular (RR).}}
\end{{table}}
""".format(
                    filename=_latex_filename,
                    depth=depth,
                    value_label=value_label,
                )
            )
        _ = f.write(r"\clearpage")